# 训练 microWakeWord 模型

本 notebook 带你训练一个基础的 microWakeWord 模型，面向高级用户，作为**起点**使用。请使用 Python 3.10。

它可以在 Google Colab 上运行，但速度远慢于本地 GPU。如果只能用 Colab，务必将运行时类型改为 GPU；即便如此仍然很慢！

**TTS 来源：** Bryce Beattie 的 Piper Clean 100，基于 LibriTTS-R train-clean-100（CC BY 4.0）从零训练。运行归档中包含来源、许可证，以及所用权重和配置的 SHA-256。[作者发布页](https://brycebeattie.com/files/tts/) · [LibriTTS-R](https://www.openslr.org/141/)。

可选正样本：Kokoro-82M v1.0（仅用于训练）。设 `USE_KOKORO = True` 后，cell 5 会生成或复用音频并对比试听。详见 [数据流程](../documentation/data_pipeline.zh.md#试听并接入-kokoro)。

数据准备会下载约 15 GB 的压缩包，请至少预留 50 GB 可用磁盘空间。分卷的 FSD50K 压缩包需要系统 `zip` 命令。

训练完成后，用量化流式验证集选最终模型。原有非流式最佳权重会保留；独立 testing 只用于检验入选模型和锁定阈值。已有权重可跳过训练，直接复评。


In [1]:
# 路径和唤醒词名称集中设置；修改后请从本单元格开始重新运行。
from pathlib import Path
import os

repo_path = Path("/content/micro-wake-word").expanduser().resolve()
data_dir = repo_path / "data_v1"
wake_word_name = "hi_circle"
wake_word_text = "Hi Circle"  # Kokoro 使用的文本，与下方 Piper 音素对应
wake_word_phonemes = "hˈaɪ sˈɜːkəl"
hey_hi_phonemes = "hˈeɪ hˈaɪ"
tts_version = "clean100_v1"
positive_dataset_name = f"{tts_version}_three_speeds"
sample_root = repo_path / "generated_samples" / wake_word_name
preview_dir = sample_root / f"{positive_dataset_name}_preview"
sample_dir = sample_root / positive_dataset_name
clean100_model_dir = data_dir / "clean100_model"
features_dir = repo_path / "features_v1"
positive_dir = features_dir / f"positive_{wake_word_name}_{positive_dataset_name}"
USE_KOKORO = False  # 设为 True：cell 5 缺音频时生成，随后只用于训练
kokoro_dataset_name = "kokoro_v1_wide_speeds_r2"
kokoro_audio_dir = sample_root / kokoro_dataset_name  # 可改到 /content/drive/MyDrive 下保存 WAV
kokoro_features_dir = features_dir / f"positive_{wake_word_name}_{kokoro_dataset_name}"
hey_hi_dataset_version = "clean100_v1"
# 可指向持久存储中已有的 raw 目录，保留 WAV 和 generation.json。
hey_hi_audio_dir = repo_path / "generated_datasets" / f"hey_hi_negative_{hey_hi_dataset_version}" / "raw"
hey_hi_features_dir = features_dir / "negative" / f"hey_hi_{hey_hi_dataset_version}"
hey_hi_context_frames = 204  # 与下方 mixednet 结构及 clip_duration_ms=1500 配套
run_suffix = f"_{tts_version}_hey_hi"
if USE_KOKORO:
    run_suffix += f"_{kokoro_dataset_name}"
feature_sources_path = features_dir / (f"feature_sources{run_suffix}.json" if USE_KOKORO else "feature_sources.json")
train_dir = repo_path / "trained_models" / f"{wake_word_name}_licensed_inputs{run_suffix}"
config_path = repo_path / f"training_parameters_{wake_word_name}{run_suffix}.yaml"
archive_path = repo_path.parent / f"{wake_word_name}{run_suffix}_run.tar.gz"
data_root = data_dir / "prepared"

# CHiME-6 真实远场 ambient 测试集：prepared/ 复用优先；压缩包与其它数据集放在同一目录。
chime6_dir = repo_path / "chime6_v1"  # 与命令行 --output-dir 默认值一致
chime6_features_dir = features_dir / "negative" / "ambient_chime6"

# Colab 首次运行时获取仓库。
import subprocess
if not repo_path.exists():
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/Seeker32/micro-wake-word.git", str(repo_path)], check=True)
os.chdir(repo_path)
assert (repo_path / "microwakeword/audio/licensed_data.py").is_file(), f"找不到仓库：{repo_path}"

if not (repo_path / "microwakeword/audio/piper_clean100.py").is_file():
    raise FileNotFoundError("请将仓库更新到包含 microwakeword/audio/piper_clean100.py 的版本。")

if USE_KOKORO and not (repo_path / "microwakeword/audio/kokoro_v1.py").is_file():
    raise FileNotFoundError("请先把新版 microwakeword/audio/kokoro_v1.py 同步到 Colab 仓库。")


In [2]:
# 安装 microWakeWord。安装完成后请重启会话。
import platform
from pathlib import Path

if platform.system() == "Darwin":
    # 在 macOS 上从 fork 安装 `pymicro-features`，以支持本地编译
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

%pip install -e "{repo_path}"

Obtaining file:///content/micro-wake-word
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 69.3 MB/s  0:00:00 eta 0:00:01
  Building editable for microwakeword (pyproject.toml) ... done
  Created wheel for microwakeword: filename=microwakeword-0.1.0-0.editable-py3-none-any.whl size=10166 sha256=a9009df809f982a50840cb7329ceaeadfb86989cffe25470bb4696309e57664d
  Stored in directory: /tmp/pip-ephem-wheel-cache-canhtk41/wheels/37/04/57/bffe1d22e8e88eedae5f8d49ca012354573c3c6f545cede40b
Successfully built microwakeword
  Attempting uninstall: soxr
    Found existing installation: soxr 1.1.0
    Uninstalling soxr-1.1.0:
      Successfully uninstalled soxr-1.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [microwakeword]0m [ai-edge-litert]


In [3]:
# 先安装 Piper，再用 CUDA 版替换其 CPU 版 ONNX Runtime。
%pip install "piper-tts==1.3.0"
%pip uninstall -y onnxruntime onnxruntime-gpu
%pip install "onnxruntime-gpu[cuda,cudnn]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 34.9 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 77.1 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [piper-tts]/2 [piper-tts]
Found existing installation: onnxruntime 1.32.0
Uninstalling onnxruntime-1.32.0:
  Successfully uninstalled onnxruntime-1.32.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.4/251.4 MB 24.3 MB/s  0:00:10m0:00:0100:01


In [4]:
import onnxruntime as ort

ort.preload_dlls()
print(ort.get_available_providers())


['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [ ]:
from IPython.display import Audio, display
from microwakeword.audio.piper_clean100 import (
    load_clean100, prepare_clean100_samples, write_clean100_preview,
)

clean100_voice = load_clean100(clean100_model_dir)
sample_path = preview_dir / "0.wav"
write_clean100_preview(clean100_voice, sample_path, wake_word_phonemes)
print("Piper Clean 100")
display(Audio(filename=str(sample_path), autoplay=True))

# 输出目录在 Drive 时提前挂载；虚拟环境仍留在运行时的本地磁盘。
drive_root = Path("/content/drive")
if USE_KOKORO and kokoro_audio_dir.is_relative_to(drive_root) and not (drive_root / "MyDrive").is_dir():
    from google.colab import drive

    drive.mount(str(drive_root))

# 优先试听当前批次；启用后才会自动生成。
pilot_preview = sample_root / "kokoro_v1_pilot" / "listen_positive.wav"
if USE_KOKORO or (kokoro_audio_dir / "generation.json").is_file():
    from microwakeword.audio.kokoro_v1 import generate_in_venv, load_training_manifest, speed_preview_wav

    if not kokoro_audio_dir.exists():
        generate_in_venv(repo_path, data_dir / "kokoro_venv", wake_word_text, kokoro_audio_dir)
    kokoro_preview = load_training_manifest(kokoro_audio_dir, wake_word_text)
    for prefix in ("af", "am", "bf", "bm"):
        voice = next((s["voice"] for s in kokoro_preview["samples"] if s["voice"].startswith(prefix)), None)
        samples = sorted((s for s in kokoro_preview["samples"] if s["voice"] == voice),
                         key=lambda s: s["speed"])
        if samples:
            speeds = " → ".join(f"{s['speed']:g}×" for s in samples)
            print(f"Kokoro v1.0 · {voice} · 由慢到快：{speeds}")
            display(Audio(data=speed_preview_wav(kokoro_audio_dir, samples)))
elif wake_word_text == "Hi Circle" and pilot_preview.is_file():
    print("Kokoro v1.0 · Hi Circle 试听小样（美/英男女声，各三档语速）")
    display(Audio(filename=str(pilot_preview)))
else:
    print("Kokoro 未启用。要生成并试听，请在顶部设置 USE_KOKORO = True 后重新运行到本单元格。")


In [6]:
# 每档 1000 条正样本。每条使用一个 Clean 100 说话人 ID。
counts = prepare_clean100_samples(clean100_voice, sample_dir, wake_word_phonemes)
print(f"原始正样本：{counts}，共 {sum(counts.values())} 条。")
print(f"试听各档：{sample_dir}/{{fast,normal,slow}}/0.wav")


clean100_v1_three_speeds · fast 已生成 1000 条 WAV
clean100_v1_three_speeds · normal 已生成 1000 条 WAV
clean100_v1_three_speeds · slow 已生成 1000 条 WAV
原始正样本：{'fast': 1000, 'normal': 1000, 'slow': 1000}，共 3000 条。
试听各档：/content/micro-wake-word/generated_samples/hi_circle/clean100_v1_three_speeds/{fast,normal,slow}/0.wav


### 首次生成或复用 hey hi 困难负样本

首次运行时，下一单元格用 Clean 100 合成快、正常、慢三档 `hey hi`，每档 1,000 条，并逐档按 80/10/10 划分。之后运行会校验并复用 `hey_hi_audio_dir` 下的 WAV 和 `generation.json`。

跨会话或跨 notebook 复用时，保留整个 `raw/` 目录，并在顶部把 `hey_hi_audio_dir` 指向同一份持久目录。特征也会在 `hey_hi_features_dir` 下校验后复用。修改 `hey_hi_phonemes` 时，请更新 `hey_hi_dataset_version` 和 `tts_version`，为新数据创建独立的缓存和训练目录。


In [7]:
# 首次生成 hey hi，已有数据只校验和复用。
reuse_hey_hi = hey_hi_audio_dir.is_dir()
# 与下方特征导出使用同样的 10 ms 步长和 0.2 秒尾部留白。
max_duration_s = ((hey_hi_context_frames - 1) * 10 - 200) / 1000
negative_counts = prepare_clean100_samples(
    clean100_voice, hey_hi_audio_dir, hey_hi_phonemes,
    hard_negative_max_duration_s=max_duration_s,
)
action = "复用" if reuse_hey_hi else "生成"
print(f"已{action} Clean 100 hey hi 困难负样本：{negative_counts}")


raw · fast 已生成 1000 条 WAV
raw · normal 已生成 1000 条 WAV
raw · slow 已生成 1000 条 WAV
已生成 Clean 100 hey hi 困难负样本：{'training': 2400, 'validation': 300, 'testing': 300}


In [8]:
# 读取原有划分的 hey hi WAV，并在训练端生成特征。
from microwakeword.audio.hard_negatives import prepare_hard_negative_features

hey_hi_lineage = prepare_hard_negative_features(
    hey_hi_audio_dir, hey_hi_features_dir,
    context_frames=hey_hi_context_frames,
)
print("hey hi 各集合特征数量:", hey_hi_lineage["counts"])


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

hey hi 各集合特征数量: {'training': 2400, 'validation': 300, 'testing': 300}


In [9]:
# （可选）从 Google Drive 预置数据集压缩包，替代下一单元格约 15 GB 的原始下载。
# 原理：prepare_licensed_data() 把压缩包缓存在 data_dir/downloads，download() 见到同名文件就跳过下载（随后按发布版 MD5 校验）。
# 因此只要压缩包与官方下载逐字节一致并保持原文件名，复制进缓存目录即可跳过 Zenodo/OpenSLR 的慢速下载。
# 需要在 DRIVE_ARCHIVE_DIR 指向的 Drive 文件夹里放好以下 8 个压缩包（文件名不变）：
#   FSD50K.eval_audio.zip 与 FSD50K.eval_audio.z01（同一分卷的两部分，缺一不可）、
#   FSD50K.metadata.zip、FSD50K.doc.zip、rirs_noises.zip、
#   train-clean-100.tar.gz、dev-clean.tar.gz、test-clean.tar.gz
# 说明：
# - FSD50K 与 LibriSpeech 有官方 MD5，复制完立即核对；不一致就删除坏副本并报错，
#   避免下一单元格在解压阶段才失败。rirs_noises.zip 官方未发布 MD5，
#   由下一单元格 zipfile 的完整性检查兜底。
# - 缓存里已有的文件跳过复制，也不重复哈希（下一单元格本来就会校验）。
# - 非 Colab 环境运行时，把 DRIVE_ARCHIVE_DIR 改成本地镜像目录即可，其余逻辑不变。

import hashlib
import shutil

from microwakeword.audio.licensed_data import FSD_FILES, SPEECH_FILES

DRIVE_ARCHIVE_DIR = Path("/content/drive/MyDrive/wake-word/data_v1/downloads")

try:
    from google.colab import drive
    in_colab = True
except ModuleNotFoundError:
    in_colab = False

if in_colab:
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    archive_source = DRIVE_ARCHIVE_DIR
else:
    archive_source = DRIVE_ARCHIVE_DIR.expanduser()
    print(f"非 Colab 环境：从本地镜像目录读取 {archive_source}")

published_md5 = dict(FSD_FILES)
published_md5.update({f"{subset}.tar.gz": md5 for _, (subset, md5) in SPEECH_FILES.items()})
expected = sorted({*published_md5, "rirs_noises.zip"})

cache_dir = data_dir / "downloads"
cache_dir.mkdir(parents=True, exist_ok=True)

copied = []
for name in expected:
    target = cache_dir / name
    if target.is_file():
        print(f"已存在，跳过复制：{name}")
        continue
    source = archive_source / name
    if not source.is_file():
        raise FileNotFoundError(
            f"{archive_source} 缺少 {name}。请先上传该压缩包（保持原文件名），"
            "或跳过本单元格，让下一单元格直接从官方源下载。"
        )
    gib = source.stat().st_size / 1024 ** 3
    print(f"复制 {name}（{gib:.2f} GiB）...", flush=True)
    shutil.copyfile(source, target)
    copied.append(name)

def _md5_of(path):
    result = hashlib.md5()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            result.update(chunk)
    return result.hexdigest()

for name in copied:
    if name not in published_md5:
        continue  # rirs_noises.zip 无官方 MD5，跳过
    actual = _md5_of(cache_dir / name)
    if actual != published_md5[name]:
        (cache_dir / name).unlink()
        raise ValueError(f"{name} 的 MD5 与发布值不符（{actual}），坏副本已删除。可重试或改走直连下载。")

total_gib = sum((cache_dir / n).stat().st_size for n in expected) / 1024 ** 3
print(f"压缩包就绪：{cache_dir}（本次复制 {len(copied)} 个，缓存共 {total_gib:.2f} GiB）。")


Mounted at /content/drive
复制 FSD50K.doc.zip（0.00 GiB）...
复制 FSD50K.eval_audio.z01（3.00 GiB）...
复制 FSD50K.eval_audio.zip（2.83 GiB）...
复制 FSD50K.metadata.zip（0.01 GiB）...
复制 dev-clean.tar.gz（0.31 GiB）...
复制 rirs_noises.zip（1.22 GiB）...
复制 test-clean.tar.gz（0.32 GiB）...
复制 train-clean-100.tar.gz（5.95 GiB）...
压缩包就绪：/content/micro-wake-word/data_v1/downloads（本次复制 8 个，缓存共 13.64 GiB）。


In [10]:
# 准备增强素材与负样本音频，并记录每个文件的来源。
# 首次运行会下载约 15 GB；之后再运行会校验已准备音频的哈希。
# 只选用 CC0/CC-BY 的 FSD50K 片段和模拟生成的 OpenSLR 28 RIR。
# 替换用的语音负样本来自 LibriSpeech。
# TTS 来源与产品依赖不在此次数据审计范围内。

from microwakeword.audio.licensed_data import prepare_licensed_data

data_root = prepare_licensed_data(str(data_dir))
print(f"Audio: {data_root}")
print(f"License manifest: {data_root / 'manifest.csv'}")
print(f"Attribution: {data_root / 'ATTRIBUTION.md'}")


Downloaded FSD50K.html: 0.25 MiB
Downloaded SLR28.html: 0.00 MiB
Downloaded SLR12.html: 0.01 MiB
Downloaded license-6cccfd94e1b0d407.txt: 0.02 MiB
Downloaded license-261eaede4865b257.txt: 0.01 MiB
Downloaded license-54895d592300bdd2.txt: 0.02 MiB
Downloaded license-f2036f817d221f2a.txt: 0.01 MiB
Audio: /content/micro-wake-word/data_v1/prepared
License manifest: /content/micro-wake-word/data_v1/prepared/manifest.csv
Attribution: /content/micro-wake-word/data_v1/prepared/ATTRIBUTION.md


In [11]:
# 设置数据增强。
# 想提升模型效果，可调整这些参数，并引入更多来源的背景片段。

from pathlib import Path

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

if not (data_root / "manifest.json").is_file():
    raise FileNotFoundError(
        f"No prepared data at {data_root.resolve()}. "
        "Run the data preparation cell from the repository directory first."
    )

clips = Clips(input_directory=str(sample_dir),
              file_pattern='*/*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
print(f"已载入 {len(clips.clips)} 条原始正样本。")
# 各 split 使用独立的背景池，避免留出音频混入训练增强。
augmenters = {
    split: Augmentation(
        augmentation_duration_s=3.2,
        augmentation_probabilities={
            "SevenBandParametricEQ": 0.1,
            "TanhDistortion": 0.1,
            "PitchShift": 0.1,
            "BandStopFilter": 0.1,
            "AddColorNoise": 0.1,
            "AddBackgroundNoise": 0.75,
            "Gain": 1.0,
            "RIR": 0.5,
        },
        impulse_paths=[str(data_root / "rirs")],
        background_paths=[str(data_root / "noise" / split)],
        background_min_snr_db=-5,
        background_max_snr_db=10,
        min_jitter_s=0.195,
        max_jitter_s=0.205,
    )
    for split in ("training", "validation", "testing")
}
augmenter = augmenters["training"]


已载入 3000 条原始正样本。


In [12]:
# 随机抽取一条片段做增强并回放，确认增强效果正常

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, str(repo_path / "augmented_clip.wav"))

Audio(str(repo_path / "augmented_clip.wav"), autoplay=True)

In [13]:
# 增强样本并保存训练、验证、测试集。
# 用同样的方式生成验证集和测试集，会让模型在基准上的表现好于真实使用。
# 改用真实录音，或用不同 TTS 引擎合成的样本，基准结果可能更接近真实情况。

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = str(positive_dir)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

splits = ["training", "validation", "testing"]
if USE_KOKORO and all((positive_dir / split / "wakeword_mmap").is_dir() for split in splits):
    print("复用 Piper 特征，保持对照组的正样本划分和增强结果。")
    splits = []
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,    # 窗长为 30 ms
                                     augmenter=augmenters[split],
                                     slide_frames=10,    # 反复使用同一张语谱图，每次只平移一帧，以在非流式训练/验证中模拟流式推理。
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenters[split],
                                     slide_frames=1,    # 测试集使用流式版本模型，无需人为重复。
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

if USE_KOKORO:
    from microwakeword.audio.kokoro_v1 import KokoroClips, MAX_DURATION_S, load_training_manifest

    # 末尾留白最多 205 ms；保守取 1.8 秒，保证截窗仍能保住整句。
    max_kokoro_duration_s = min(MAX_DURATION_S, (hey_hi_context_frames - 1) * 0.01 - 0.205)
    kokoro_generation = load_training_manifest(kokoro_audio_dir, wake_word_text,
                                                max_duration_s=max_kokoro_duration_s)
    kokoro_clips = KokoroClips(kokoro_audio_dir, kokoro_generation)
    kokoro_spectrograms = SpectrogramGeneration(
        clips=kokoro_clips, augmenter=augmenters["training"], slide_frames=10, step_ms=10,
    )
    kokoro_training_mmap = kokoro_features_dir / "training" / "wakeword_mmap"
    kokoro_training_mmap.parent.mkdir(parents=True, exist_ok=True)
    RaggedMmap.from_generator(
        out_dir=str(kokoro_training_mmap),
        sample_generator=kokoro_spectrograms.spectrogram_generator(repeat=2),
        batch_size=100,
        verbose=True,
    )
    print(f"Kokoro · {len(kokoro_generation['samples'])} 条原始正样本（仅训练）")


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [14]:
# 用已审计的原始音频构建替换用的负样本特征。
# 各来源的 split 划分保持不变。ambient 评估使用 ≥6 秒的原始片段，不做人工重复，作为真实设备房间录音的近似。

import json
from pathlib import Path
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.licensed_data import digest

negative_root = features_dir / "negative"
for corpus, pattern in (("speech", "*.flac"), ("noise", "*.wav")):
    for split in ("training", "validation", "testing"):
        source_dir = data_root / corpus / split
        negative_clips = Clips(
            input_directory=str(source_dir),
            file_pattern=pattern,
            repeat_clip_min_duration_s=6,
        )
        generator = SpectrogramGeneration(
            clips=negative_clips,
            step_ms=10,
            split_spectrogram_duration_s=5,
        )
        out_dir = negative_root / corpus / split
        out_dir.mkdir(parents=True, exist_ok=True)
        RaggedMmap.from_generator(
            out_dir=str(out_dir / "negative_mmap"),
            sample_generator=generator.spectrogram_generator(),
            batch_size=100,
            verbose=True,
        )
        if split != "training":
            ambient_clips = Clips(
                input_directory=str(source_dir),
                file_pattern=pattern,
                min_clip_duration_s=6,
            )
            if len(ambient_clips.clips) == 0:
                raise ValueError(f"No original clips >=6s for ambient evaluation: {source_dir}")
            ambient_generator = SpectrogramGeneration(clips=ambient_clips, step_ms=10)
            out_dir = negative_root / "ambient" / f"{split}_ambient"
            out_dir.mkdir(parents=True, exist_ok=True)
            RaggedMmap.from_generator(
                out_dir=str(out_dir / f"{corpus}_mmap"),
                sample_generator=ambient_generator.spectrogram_generator(),
                batch_size=100,
                verbose=True,
            )

# 把派生特征与原始文件的许可证清单精确关联起来。
lineage = {
    "audio_manifest": str((data_root / "manifest.json").resolve()),
    "audio_manifest_sha256": digest(data_root / "manifest.json"),
    "negative_features": str(negative_root),
    "positive_features": str(positive_dir),
    "positive_generation_manifest": str(sample_dir / "generation.json"),
    "processing": "16 kHz mono on access; 10 ms feature step; 5s negative chunks; "
                  "short classification negatives repeated to >=6s; ambient clips not repeated",
    "positive_augmentation": {split: {
        "background_pool": str(data_root / "noise" / split),
        "rir_pool": str(data_root / "rirs"),
    } for split in ("training", "validation", "testing")},
    "pending": "Dependency audit and real-device evaluation",
}
lineage["hard_negatives"] = {"hey_hi": hey_hi_lineage}
if USE_KOKORO:
    lineage["kokoro_positive"] = {
        "features": str(kokoro_features_dir),
        "generation_manifest": str(kokoro_audio_dir / "generation.json"),
        "training_wavs": len(kokoro_generation["samples"]),
    }
feature_sources_path.write_text(json.dumps(lineage, indent=2) + "\n")


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

1003005

### CHiME-6 真实远场 ambient 评估集

误唤醒只有在真实房间录音上量才有意义：FSD50K 的噪声片段很干净，LibriSpeech 是近讲朗读，两者都会低估家庭场景。这里接入 CHiME-6（OpenSLR SLR150，CC BY-SA 4.0）——与 DiPCo/CHiME-5 同源的家庭多人晚餐对话，取每个房间参考阵列的 CH1，切成 10 分钟连续片段。

- 只做 ambient 评估：`sampling_weight = 0`，不参与训练采样，也不做重复或切块，保持原始连续音频。
- dev → `validation_ambient`（4.46 h），eval → `testing_ambient`（15.62 h），沿用官方 session 划分，不跨 split。
- 数据来源优先级：`chime6_v1/prepared`（校验通过即复用）→ 与其它数据集同一目录里的 `chime6_v1.zip`（`data_v1/downloads`，或 Drive 上那个下载目录；解压后是 `prepared/`，也接受已解压目录）→ OpenSLR 流式下载（可断点续传，约 11.9 GB，峰值磁盘约 12 GB，成功后只留 prepared）。
- 内存：`validation_ambient` 会在训练时整体载入（4.46 h ≈ 1.2 GB）。内存吃紧时可改成 `prepare_ambient_features(..., splits=("testing",))` 只建 testing 侧，训练期的 `validation_ambient` 就回落到 FSD50K/LibriSpeech。


In [15]:
# 接入 CHiME-6：复用 prepared/ → 同目录的 chime6_v1.zip → 才从 OpenSLR 流式下载（断点续传）。
# 只构建 ambient 特征（validation_ambient / testing_ambient），不参与训练采样。
# 压缩包不用单独配置路径：与其它数据集一样放进 data_v1/downloads（或 Drive 的同一个下载目录）。
from microwakeword.audio.chime6 import ensure_chime6, prepare_ambient_features

chime6_source_dirs = [data_dir / "downloads"]
if "DRIVE_ARCHIVE_DIR" in globals():  # 上面的 Drive 单元格已挂载并定义该目录
    chime6_source_dirs.append(DRIVE_ARCHIVE_DIR)
chime6_prepared_dir = ensure_chime6(chime6_dir, source_dirs=chime6_source_dirs)
chime6_lineage = prepare_ambient_features(chime6_prepared_dir, chime6_features_dir)
print(f"CHiME-6 特征目录：{chime6_features_dir}")
print(f"片段数 {len(chime6_lineage['clips'])}，时长(小时)："
      f"{ {split: round(hours, 2) for split, hours in chime6_lineage['hours'].items()} }")


Restoring CHiME-6 from /content/drive/MyDrive/wake-word/data_v1/downloads/chime6_v1.zip
Unpacking chime6_v1.zip into /content/micro-wake-word/chime6_v1
Built ambient features in /content/micro-wake-word/features_v1/negative/ambient_chime6
CHiME-6 特征目录：/content/micro-wake-word/features_v1/negative/ambient_chime6
片段数 123，时长(小时)：{'validation': 4.46, 'testing': 15.62}


In [16]:
# 保存控制训练过程的 YAML 配置。
# 这些超参数对模型质量影响很大。
# 可尝试调整采样权重、惩罚权重，并增加训练步数。

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = str(train_dir)


# 每个 feature_dir 至少应包含以下某一种结构的文件夹：
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight：该集合在批次中被抽到语谱图的权重
#  penalty_weight：该集合预测错误的惩罚权重
#  truth：该集合是正样本（True）还是负样本（False）
#  truncation_strategy：集合中的语谱图长于训练所需时如何截断
#       - random：从整张语谱图中随机取一段，适合长负样本
#       - truncate_start：去掉语谱图开头
#       - truncate_end：去掉语谱图结尾
#       - split：把较长的语谱图按 100 ms 偏移切成多张，仅用于 ambient 集合

config["features"] = [
    {
        "features_dir": str(positive_dir),
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": str(features_dir / "negative" / "speech"),
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": str(features_dir / "negative" / "noise"),
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    { # 仅用于验证和测试
        "features_dir": str(features_dir / "negative" / "ambient"),
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]



# 独立采样 hey hi 困难负样本，保留普通语音/噪声负样本。
config["features"].insert(3, {
    "features_dir": str(hey_hi_features_dir),
    "sampling_weight": 2.0,
    "penalty_weight": 1.0,
    "truth": False,
    "truncation_strategy": "truncate_start",  # 短语在末尾，避免随机采到静音
    "type": "mmap",
})

# 分类预测错误的惩罚权重；列表元素与训练步数对应
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

if USE_KOKORO:
    if not (kokoro_features_dir / "training" / "wakeword_mmap").is_dir():
        raise FileNotFoundError(f"找不到 Kokoro 训练特征：请先运行 cell 14（{kokoro_features_dir}）。")
    piper_positive = config["features"][0]
    piper_positive["sampling_weight"] = 1.8
    config["features"].append({
        **piper_positive, "features_dir": str(kokoro_features_dir), "sampling_weight": 0.2,
    })

# CHiME-6 真实远场家庭录音：只进 validation_ambient / testing_ambient，
# sampling_weight=0 表示不参与训练采样；它的误唤醒率会并入 ambient 汇总指标，
# CHiME-6 单独的数字见后面的 evaluate_ambient_model 单元格。
config["features"].append({
    "features_dir": str(chime6_features_dir),
    "sampling_weight": 0.0,
    "penalty_weight": 1.0,
    "truth": False,
    "truncation_strategy": "split",
    "type": "mmap",
})

# 每次迭代的训练步数；其它多项设置也以列表形式与不同步数对应
config["training_steps"] = [10000]
config["learning_rates"] = [0.001]  # Adam 优化器的学习率；列表元素与训练步数对应
config["batch_size"] = 128

config["time_mask_max_size"] = [0]  # SpecAugment 参数；列表元素与训练步数对应
config["time_mask_count"] = [0]  # SpecAugment 参数；列表元素与训练步数对应
config["freq_mask_max_size"] = [0]  # SpecAugment 参数；列表元素与训练步数对应
config["freq_mask_count"] = [0]  # SpecAugment 参数；列表元素与训练步数对应

config["eval_step_interval"] = (500)  # 每隔这么多步评估一次验证集
config["clip_duration_ms"] = (1500)  # 流式模型可接受的最长唤醒词时长


# 最佳模型权重先按 minimization_metric 最小化挑选，且要求已低于 target_minimization；
# 达标之后改为取 maximization_metric 的最大值。把 'minimization_metric' 设为 None 则只看最大化指标。
# 可用指标：
#   - "loss"：验证集交叉熵误差
#   - "accuracy"：验证集准确率
#   - "recall"：验证集召回率
#   - "precision"：验证集精确率
#   - "false_positive_rate"：验证集误唤醒率
#   - "false_negative_rate"：验证集漏唤醒率
#   - "ambient_false_positives"：拆分的 validation_ambient 集合误唤醒次数
#   - "ambient_false_positives_per_hour"：拆分的 validation_ambient 集合每小时误唤醒估计次数
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # 设为 None 即禁用

config["maximization_metric"] = "average_viable_recall"

# 当前命令的卷积结构共丢弃 154 帧；模型结构改变时需同步检查此值。
expected_context = 1 + (config["clip_duration_ms"] - 30) // (3 * config["window_step_ms"]) + 154
if hey_hi_context_frames != expected_context:
    raise ValueError(f"hey_hi_context_frames 应与训练上下文一致: {expected_context}")

with open(config_path, "w") as file:
    documents = yaml.dump(config, file)

In [ ]:
# 训练与选型分开：此步只训练并保存快照，不读测试集做评估。
import subprocess
import sys

RUN_TRAINING = True  # 复评已有权重时设为 False
STREAMING_FAPH_BUDGET = 0.1  # 验证集上的每小时误唤醒预算；不是发布验收标准

if RUN_TRAINING:
    subprocess.run([
        sys.executable, "-m", "microwakeword.model_train_eval",
        "--training_config", str(config_path),
        "--train", "1", "--restore_checkpoint", "1", "--test_tflite_streaming_quantized", "0",
        "mixednet", "--pointwise_filters", "64,64,64,64",
        "--repeat_in_block", "1, 1, 1, 1", "--mixconv_kernel_sizes", "[5], [7,11], [9,15], [23]",
        "--residual_connection", "0,0,0,0", "--first_conv_filters", "32",
        "--first_conv_kernel_size", "5", "--stride", "3",
    ], check=True)


In [ ]:
# 量化流式选型：同一校准集、同一验证集，只有入选模型才跑独立测试。
import filecmp
import json
import math
import random
import re
import shutil
from tempfile import TemporaryDirectory
from types import SimpleNamespace

import numpy as np
import tensorflow as tf
import yaml

from microwakeword import mixednet, utils
from microwakeword.audio import ambient_eval as ae
from microwakeword.data import FeatureHandler
from microwakeword.layers import modes

class ConfigLoader(yaml.SafeLoader):
    pass

ConfigLoader.add_constructor("tag:yaml.org,2002:python/tuple",
                             lambda loader, node: tuple(loader.construct_sequence(node)))
archived = yaml.load((train_dir / "training_config.yaml").read_text(), Loader=ConfigLoader)
flags = SimpleNamespace(**archived["flags"])
if flags.model_name != "mixednet":
    raise ValueError("此 notebook 使用 MixedNet；请选择对应的训练目录。")
if not math.isfinite(STREAMING_FAPH_BUDGET) or STREAMING_FAPH_BUDGET < 0:
    raise ValueError("STREAMING_FAPH_BUDGET 必须是有限的非负数。")
# 已归档的权重使用原结构；特征从当前 notebook 的路径读取。
eval_config = {**archived, "features": config["features"]}
STRIDE, STEP_MS = eval_config["stride"], eval_config["window_step_ms"]
WINDOW, COOLDOWN = ae.WINDOW, ae.IGNORE_SLICES_AFTER_ACCEPT
SCORE_MAX = 255 * WINDOW  # 离线推理将 uint8 除以 255，再取滑动均值
TFLITE_DIR = "tflite_stream_state_internal_quant"
TFLITE_FILE = "stream_state_internal_quant.tflite"

def shortlist(directory):
    best = directory / "best_weights.weights.h5"
    if not best.is_file():
        raise FileNotFoundError(f"没有非流式最佳权重：{best}")
    snapshots = {}
    training_end = sum(eval_config["training_steps"])
    interval = eval_config["eval_step_interval"]
    for file in sorted((directory / "train").glob("*_weights_*.weights.h5"),
                       key=lambda path: (path.stat().st_mtime_ns, path.name)):
        match = re.fullmatch(r"-?\d+_weights_(\d+)\.weights\.h5", file.name)
        if match:
            step = int(match[1])
            if 0 < step <= training_end and (step % interval == 0 or step == training_end):
                snapshots[step] = file
    best_step = next((step for step, file in sorted(snapshots.items())
                      if filecmp.cmp(best, file, shallow=False)), None)
    candidates = [(best_step, best)]
    if snapshots:
        end = max(snapshots)
        # 简化：只复评少量快照；需找全程最优时扩大候选，不在每次训练验证时量化。
        late = min(snapshots, key=lambda step: (abs(4 * step - 3 * end), step))
        for step in sorted({min(snapshots), late, end} - {best_step}):
            file = snapshots[step]
            if not any(filecmp.cmp(file, other, shallow=False) for _, other in candidates):
                candidates.append((step, file))
    return candidates

def restore_validation_tracks(rows):
    # basic 的非流式 validation 对每条录音写入 10 个顺序错位窗。
    shifts = 10
    if not len(rows) or len(rows) % shifts:
        raise ValueError("validation 正样本必须是完整的 10 窗组；请使用本 notebook 生成的特征。")
    tracks = []
    for start in range(0, len(rows), shifts):
        first, last = rows[start], rows[start + shifts - 1]
        if len(first) < shifts or not np.array_equal(first[shifts - 1:], last[:1 - shifts]):
            raise ValueError(f"validation 的第 {start // shifts + 1} 组错位窗不匹配。")
        tracks.append(np.concatenate((first, last[1 - shifts:])))
    return tracks

def load_streaming_split(split):
    positives = ae.open_mmap(positive_dir / split / "wakeword_mmap")
    ambient = ae.find_ambient_sets(features_dir / "negative/ambient", chime6_features_dir,
                                  mode=f"{split}_ambient")
    if not {"speech", "noise"} <= ambient.keys():
        raise ValueError(f"{split} 缺少 LibriSpeech 或 FSD50K 的背景特征。")
    if split == "validation":
        positives = restore_validation_tracks(positives)
    return positives, ambient

def score_bins(values):
    return np.rint(np.asarray(values) * SCORE_MAX).astype(np.int16)

def stream_scores(model_path, split_data):
    positives, sources = split_data
    player = ae.StreamPlayer(model_path, stride=STRIDE)
    peaks = ae.evaluate_tracks(positives, player, window=WINDOW,
                               skip_first=ae.POSITIVE_SKIP_FIRST)["peaks"]
    tracks = {}
    for name, paths in sources.items():
        tracks[name] = []
        for path in paths:
            values = ae.evaluate_tracks(ae.open_mmap(path), player, window=WINDOW)["probabilities"]
            tracks[name].extend(score_bins(track) for track in values)
        print(f"{name}: {len(tracks[name])} 段背景音", flush=True)
    return score_bins(peaks), tracks

def count_accepts(tracks, cutoff):
    total = 0
    for track in tracks:
        above = np.flatnonzero(track[COOLDOWN - 1:] > cutoff)
        index = 0
        while index < len(above):
            total += 1
            index = int(np.searchsorted(above, above[index] + COOLDOWN))
    return total

def lowest_cutoff(tracks, max_accepts):
    # 固定 cooldown 的贪心计数是最多可取的不重叠事件数；阈值越高，候选事件只会减少。
    lo, hi = 0, SCORE_MAX
    while lo < hi:
        mid = (lo + hi) // 2
        if count_accepts(tracks, mid) <= max_accepts:
            hi = mid
        else:
            lo = mid + 1
    return lo

def hours(tracks):
    return ae.track_hours(tracks, stride=STRIDE, step_s=STEP_MS / 1000)

def point(scored, cutoff):
    peaks, sources = scored
    by_source = {name: {"ambient_hours": hours(tracks),
                        "false_accepts": count_accepts(tracks, cutoff)}
                 for name, tracks in sources.items()}
    duration = sum(row["ambient_hours"] for row in by_source.values())
    accepts = sum(row["false_accepts"] for row in by_source.values())
    if not len(peaks) or not duration:
        raise ValueError("正样本或背景评估集为空。")
    missed = int(np.count_nonzero(peaks <= cutoff))
    return {"cutoff_bin": int(cutoff), "cutoff": float(cutoff / SCORE_MAX),
            "frr": missed / len(peaks), "missed": missed, "positives": len(peaks),
            "faph": accepts / duration, "false_accepts": accepts, "ambient_hours": duration,
            "ambient_sources": by_source}

def export_candidate(weights, directory, calibration):
    class SharedCalibration:
        def get_data(self, mode, batch_size, features_length):
            # 转换器会给第一条特征写入 min/max，故每个候选取同一数组的独立副本。
            return calibration.copy(), None, None

    tf.keras.backend.clear_session()
    model = mixednet.model(flags, tuple(eval_config["training_input_shape"]), batch_size=1)
    model.load_weights(weights)
    candidate = {**eval_config, "train_dir": str(directory)}
    utils.convert_model_saved(model, candidate, "stream_state_internal",
                              modes.Modes.STREAM_INTERNAL_STATE_INFERENCE)
    utils.convert_saved_model_to_tflite(
        candidate, SharedCalibration(), str(directory / "stream_state_internal"),
        str(directory / TFLITE_DIR), TFLITE_FILE, quantize=True,
    )

def write_diagnostic_roc(scored, destination):
    # 延续旧报告的 0.01 阈值网格和 0–2 FAPH 面积；只作诊断，不参与选型。
    peaks, sources = scored
    tracks = [track for group in sources.values() for track in group]
    duration = hours(tracks)
    cutoffs = np.linspace(0, 1, 101)
    bins = np.floor(cutoffs * SCORE_MAX + 1e-9).astype(int)
    faph = np.array([count_accepts(tracks, k) / duration for k in bins])
    frr = np.array([np.count_nonzero(peaks <= k) / len(peaks) for k in bins])
    curve = ae.roc_curve(faph, frr, cutoffs)
    rows = [f"AUC {curve['auc']:.5f}"]
    rows.extend(f"Cutoff {cutoff:.2f}: frr={miss:.4f}; faph={rate:.3f}"
                for cutoff, miss, rate in zip(curve["cutoffs"], curve["frr"], curve["faph"]))
    destination.write_text("\n".join(rows) + "\n")

def select_streaming_checkpoint():
    candidates = shortlist(train_dir)
    print("候选：", ", ".join(f"{step if step is not None else '非流式最佳'} ({path.name})"
                             for step, path in candidates))
    validation = load_streaming_split("validation")
    random.seed(42)
    np.random.seed(42)
    calibration_config = eval_config
    if USE_KOKORO:
        # 对照实验沿用 Piper-only 校准分布，不让校准也成为变量。
        calibration_config = {**eval_config, "features": [
            {**row, "sampling_weight": 2.0} if row["features_dir"] == str(positive_dir) else row
            for row in eval_config["features"] if row["features_dir"] != str(kokoro_features_dir)
        ]}
    calibration, _, _ = FeatureHandler(calibration_config).get_data(
        "training", 500, features_length=eval_config["spectrogram_length"])
    report = {"faph_budget": STREAMING_FAPH_BUDGET, "score_denominator": SCORE_MAX,
              "comparison": "score > cutoff", "calibration_windows": len(calibration),
              "validation": [], "winner": None, "held_out_test": None}
    report_path = train_dir / "streaming_checkpoint_selection.json"
    with TemporaryDirectory(prefix="streaming_selection_", dir=train_dir) as tmp:
        staging = Path(tmp)
        for i, (step, weights) in enumerate(candidates):
            print(f"量化并验证 {weights.name}", flush=True)
            directory = staging / str(i)
            export_candidate(weights, directory, calibration)
            scores = stream_scores(directory / TFLITE_DIR / TFLITE_FILE, validation)
            tracks = [track for group in scores[1].values() for track in group]
            duration = hours(tracks)
            cutoff = lowest_cutoff(tracks, math.floor(STREAMING_FAPH_BUDGET * duration))
            row = {"step": step, "weights": str(weights.relative_to(train_dir)), **point(scores, cutoff)}
            report["validation"].append(row)
            print(f"阈值 {row['cutoff']:.6f} · 漏唤醒 {row['missed']}/{row['positives']} · "
                  f"误唤醒 {row['false_accepts']} 次 / {row['ambient_hours']:.2f} h", flush=True)
            del scores, tracks
        eligible = [i for i, row in enumerate(report["validation"])
                    if row["faph"] <= STREAMING_FAPH_BUDGET and row["frr"] < 1]
        if not eligible:
            report_path.write_text(json.dumps(report, indent=2) + "\n")
            raise RuntimeError("没有候选在预算内检出正样本；选型失败，未替换现有模型。")
        best = min(eligible, key=lambda i: (report["validation"][i]["frr"],
                                           report["validation"][i]["faph"]))
        winner = report["winner"] = report["validation"][best]
        chosen_model = staging / str(best) / TFLITE_DIR / TFLITE_FILE
        print(f"入选 {winner['weights']} · 锁定阈值 {winner['cutoff']:.6f}", flush=True)
        scores = stream_scores(chosen_model, load_streaming_split("testing"))
        held_out = report["held_out_test"] = point(scores, winner["cutoff_bin"])
        hard_tracks = ae.open_mmap(hey_hi_features_dir / "testing/hey_hi_mmap")
        hard_peaks = ae.evaluate_tracks(
            hard_tracks, ae.StreamPlayer(chosen_model, stride=STRIDE),
            window=WINDOW, skip_first=ae.POSITIVE_SKIP_FIRST,
        )["peaks"]
        held_out["hey_hi_clips"] = len(hard_peaks)
        held_out["hey_hi_false_triggers"] = int(np.count_nonzero(
            score_bins(hard_peaks) > winner["cutoff_bin"]))
        report["held_out_test_meets_faph_budget"] = held_out["faph"] <= STREAMING_FAPH_BUDGET
        write_diagnostic_roc(scores, chosen_model.with_name("tflite_streaming_roc.txt"))
        # 发布被验证的同一份 TFLite，不重新校准；保留原 best_weights 的非流式含义。
        for folder in ("stream_state_internal", TFLITE_DIR):
            shutil.copytree(staging / str(best) / folder, train_dir / folder, dirs_exist_ok=True)
        shutil.copy2(candidates[best][1], train_dir / "best_streaming_weights.weights.h5")
        report_path.write_text(json.dumps(report, indent=2) + "\n")
    if winner["ambient_hours"] * STREAMING_FAPH_BUDGET < 1:
        print("注意：当前验证集出现 1 次误唤醒就超过预算，只能比较零观测误唤醒工作点。")
    print(f"独立测试：漏唤醒 {held_out['missed']}/{held_out['positives']} · "
          f"误唤醒 {held_out['false_accepts']} 次 / {held_out['ambient_hours']:.2f} h · "
          f"hey hi {held_out['hey_hi_false_triggers']}/{held_out['hey_hi_clips']} · "
          f"误唤醒预算{'达标' if report['held_out_test_meets_faph_budget'] else '未达标（阈值保持不变）'}")
    print(f"选型报告：{report_path}")
    return report

streaming_selection = select_streaming_checkpoint()


In [ ]:
# 单独评估 hey hi 测试集：每条音频触发与否，不与 ambient 的每小时误唤醒率混算。
import json
from microwakeword.audio.hard_negatives import evaluate_hard_negative_model

hey_hi_report = evaluate_hard_negative_model(
    train_dir / "tflite_stream_state_internal_quant/stream_state_internal_quant.tflite",
    hey_hi_features_dir,
)
report_path = train_dir / "hey_hi_test_results.json"
report_path.write_text(json.dumps(hey_hi_report, indent=2) + "\n", encoding="utf-8")
for result in hey_hi_report["results"]:
    print(f"hey hi 阈值 {result['cutoff']:.2f}: "
          f"误触发 {result['false_triggers']}/{hey_hi_report['sample_count']} "
          f"({result['false_trigger_rate']:.2%})")


In [ ]:
# CHiME-6 ambient 误唤醒报告：每小时误唤醒次数（与 testing_ambient 同口径），
# 并列出触发最强的片段，便于判断是电视/音乐还是家庭对话造成的误唤醒。
import json
from pathlib import Path

import numpy as np

from microwakeword.audio.chime6 import evaluate_ambient_model

ambient_model_path = (train_dir / "tflite_stream_state_internal_quant"
                      / "stream_state_internal_quant.tflite")
ambient_cutoffs = np.array([0.9, 0.95, 0.98, 0.99])
chime6_report = {
    "testing": evaluate_ambient_model(ambient_model_path, chime6_features_dir,
                                      split="testing", cutoffs=ambient_cutoffs)
}
report_path = train_dir / "chime6_ambient_results.json"
report_path.write_text(json.dumps(chime6_report, indent=2) + "\n", encoding="utf-8")
for split, report in chime6_report.items():
    print(f"{split}: {report['clip_count']} 个片段 / {report['hours']:.2f} h")
    for row in report["results"]:
        print(f"  阈值 {row['cutoff']:.2f}: {row['false_accepts_per_hour']:.2f} 次/小时"
              f"（{row['false_accepts']:.1f} 次，涉及 {row['clips_with_any_trigger']} 个片段）")
    for clip in report["loudest_clips"][:3]:
        print(f"    峰值 {clip['peak_probability']:.3f}  {clip['session']} "
              f"{clip['room']} {clip['array']} {clip['path'].rsplit('_', 1)[-1]}")
print(f"报告：{report_path}")


In [ ]:
# 打包训练结果和数据来源记录。
import shutil
import subprocess
import tempfile

marker = hey_hi_audio_dir / "generation.json"
marker_base = repo_path if marker.is_relative_to(repo_path) else hey_hi_audio_dir.parent
hey_hi_archive_inputs = [
    str((hey_hi_features_dir / "feature_sources.json").relative_to(repo_path)),
    "-C", str(marker_base), str(marker.relative_to(marker_base)),
]

kokoro_archive_inputs = [
    "-C", str(kokoro_audio_dir.parent), f"{kokoro_audio_dir.name}/generation.json",
] if USE_KOKORO else []

# CHiME-6 的来源文件放进独立前缀，避免与 licensed_data 的 prepared/ 同名。
with tempfile.TemporaryDirectory() as staging:
    provenance = Path(staging) / "chime6" / "prepared"
    provenance.mkdir(parents=True)
    for name in ("manifest.json", "manifest.csv", "ATTRIBUTION.md"):
        shutil.copy2(chime6_dir / "prepared" / name, provenance / name)
    shutil.copytree(chime6_dir / "prepared" / "sources", provenance / "sources")
    subprocess.run([
        "tar", "-czf", str(archive_path),
        "-C", str(repo_path),
        str(train_dir.relative_to(repo_path)),
        str(config_path.relative_to(repo_path)),
        str(feature_sources_path.relative_to(repo_path)),
        str((sample_dir / "generation.json").relative_to(repo_path)),
        str((chime6_features_dir / "feature_sources.json").relative_to(repo_path)),
        *hey_hi_archive_inputs,
        *kokoro_archive_inputs,
        "-C", str(staging),
        "chime6/prepared/manifest.json", "chime6/prepared/manifest.csv",
        "chime6/prepared/ATTRIBUTION.md", "chime6/prepared/sources",
        "-C", str(data_dir),
        "prepared/manifest.json", "prepared/manifest.csv", "prepared/ATTRIBUTION.md",
        "prepared/excluded_fsd.csv", "prepared/sources",
    ], check=True)
print(f"已创建：{archive_path}")
